# RAG Pipeline Walkthrough

This notebook explains the current end-to-end pipeline for the phase 1 chatbot.

Current rules:

- `data/corpus/knowledge_base/` is the retrievable knowledge corpus.
- `data/corpus/guidelines/` is not indexed as user-facing knowledge. It should guide answer style and decision logic later.
- `data/corpus/Social_Media_Advertising.csv` is structured data. It should be analyzed with tabular methods, not blindly chunked as RAG text.
- Embeddings default to `BAAI/bge-m3` on CPU.
- Vector storage uses local FAISS under `data/indexes/faiss/knowledge_base/`.


## 1. Project Setup

The code lives in `src/`, so the notebook first adds the repository root to `sys.path`.

Use this notebook from the `chatbot_llm_rag_taller` conda environment.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

WindowsPath('c:/Users/pedro/github/dmc-tp2-chatbot')

## 2. Load Settings

`IndexingSettings` centralizes the corpus path, index path, embedding provider, embedding model, and chunking parameters.

The defaults are intentionally aligned with the current project decisions.

In [2]:
from src.config.settings import IndexingSettings, RetrievalSettings

settings = IndexingSettings()
retrieval_settings = RetrievalSettings(top_k=4)

settings

IndexingSettings(corpus_path=WindowsPath('C:/Users/pedro/github/dmc-tp2-chatbot/data/corpus/knowledge_base'), index_path=WindowsPath('C:/Users/pedro/github/dmc-tp2-chatbot/data/indexes/faiss/knowledge_base'), embedding_provider='sentence_transformers', embedding_model='BAAI/bge-m3', chunk_size=800, chunk_overlap=120)

## 3. Load Knowledge Base PDFs

The loader reads PDFs only from `knowledge_base`. Each page becomes a LangChain `Document` with metadata such as source path, document name, page, document id, and corpus group.

This metadata is important because later answers need source traceability.

In [3]:
from src.loaders.pdf_loader import load_pdf_directory

documents = load_pdf_directory(settings.corpus_path)
len(documents), documents[0].metadata

c:\Users\pedro\anaconda3\envs\chatbot_llm_rag_taller\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(66,
 {'producer': 'ReportLab PDF Library - (opensource)',
  'creator': '(unspecified)',
  'creationdate': '2026-05-09T02:23:51-05:00',
  'author': 'Equipo de Analítica',
  'keywords': '',
  'moddate': '2026-05-09T02:23:51-05:00',
  'subject': '(unspecified)',
  'title': 'Instagram vs Twitter: ¿Cuál Canal Genera Mayor ROI en 2022?',
  'trapped': '/False',
  'source': 'C:\\Users\\pedro\\github\\dmc-tp2-chatbot\\data\\corpus\\knowledge_base\\A01_Instagram_vs_Twitter_¿Cuál_Canal_Genera.pdf',
  'total_pages': 1,
  'page': 0,
  'page_label': '1',
  'source_path': 'C:\\Users\\pedro\\github\\dmc-tp2-chatbot\\data\\corpus\\knowledge_base\\A01_Instagram_vs_Twitter_¿Cuál_Canal_Genera.pdf',
  'document_name': 'A01_Instagram_vs_Twitter_¿Cuál_Canal_Genera.pdf',
  'document_id': 'A01_Instagram_vs_Twitter_¿Cuál_Canal_Genera',
  'source_type': 'pdf',
  'corpus_group': 'knowledge_base'})

## 4. Split Documents Into Chunks

RAG does not embed full PDFs directly. It embeds smaller chunks so retrieval can find the most relevant passages.

Current defaults:

- `chunk_size = 800`
- `chunk_overlap = 120`

The overlap helps preserve context when an idea crosses a chunk boundary.

In [4]:
from src.splitters.recursive import split_documents

chunks = split_documents(
    documents,
    chunk_size=settings.chunk_size,
    chunk_overlap=settings.chunk_overlap,
)

len(chunks), chunks[0].metadata, chunks[0].page_content[:500]

(150,
 {'producer': 'ReportLab PDF Library - (opensource)',
  'creator': '(unspecified)',
  'creationdate': '2026-05-09T02:23:51-05:00',
  'author': 'Equipo de Analítica',
  'keywords': '',
  'moddate': '2026-05-09T02:23:51-05:00',
  'subject': '(unspecified)',
  'title': 'Instagram vs Twitter: ¿Cuál Canal Genera Mayor ROI en 2022?',
  'trapped': '/False',
  'source': 'C:\\Users\\pedro\\github\\dmc-tp2-chatbot\\data\\corpus\\knowledge_base\\A01_Instagram_vs_Twitter_¿Cuál_Canal_Genera.pdf',
  'total_pages': 1,
  'page': 0,
  'page_label': '1',
  'source_path': 'C:\\Users\\pedro\\github\\dmc-tp2-chatbot\\data\\corpus\\knowledge_base\\A01_Instagram_vs_Twitter_¿Cuál_Canal_Genera.pdf',
  'document_name': 'A01_Instagram_vs_Twitter_¿Cuál_Canal_Genera.pdf',
  'document_id': 'A01_Instagram_vs_Twitter_¿Cuál_Canal_Genera',
  'source_type': 'pdf',
  'corpus_group': 'knowledge_base',
  'chunk_index': 0,
  'chunk_id': 'A01_Instagram_vs_Twitter_¿Cuál_Canal_Genera-chunk-0000'},
 '[ANÁLISIS RÁPIDO]\n I

## 5. Create Embeddings

Embeddings convert text chunks into numerical vectors.

We use `BAAI/bge-m3` through `sentence-transformers` on CPU because it is multilingual, strong for retrieval, and realistic for this machine.

This cell loads the model. The first run may take longer if Hugging Face needs to download it.

In [5]:
from src.embeddings.factory import create_embeddings

embeddings = create_embeddings(
    settings.embedding_provider,
    settings.embedding_model,
    device="cpu",
)

type(embeddings).__name__, settings.embedding_model

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 30074.69it/s]


('SentenceTransformerEmbeddings', 'BAAI/bge-m3')

## 6. Build Or Reuse The FAISS Index

FAISS stores the chunk vectors locally and supports fast semantic search.

The current index uses maximum inner product over normalized BGE-M3 vectors. For normalized vectors, inner product is cosine similarity.

The pipeline writes a `manifest.json` next to the FAISS files. The manifest records:

- corpus hash
- embedding provider
- embedding model
- chunk size
- chunk overlap
- vector backend

If these values still match, the app can reuse the existing index instead of rebuilding it.

In [6]:
from src.pipeline import build_or_load_index
from src.vectorstores.faiss_store import index_is_valid, load_manifest

print("Index valid before load:", index_is_valid(settings))

vector_store = build_or_load_index(settings, force_rebuild=False)

print("Index path:", settings.index_path)
print("Manifest:", load_manifest(settings.index_path))

Index valid before load: True


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 39101.90it/s]


Index path: C:\Users\pedro\github\dmc-tp2-chatbot\data\indexes\faiss\knowledge_base
Manifest: IndexManifest(corpus_path='C:\\Users\\pedro\\github\\dmc-tp2-chatbot\\data\\corpus\\knowledge_base', corpus_hash='4276e45e2417e3f71d36ce4910aca0e8e004644e86347e8a13556e2a14d27a56', embedding_provider='sentence_transformers', embedding_model='BAAI/bge-m3', chunk_size=800, chunk_overlap=120, vector_backend='faiss', distance_strategy='max_inner_product')


### Confirm The FAISS Similarity Metric

This project uses `MAX_INNER_PRODUCT` over normalized embeddings. That is cosine similarity in practice.

The loaded FAISS index should be `IndexFlatIP`, where `IP` means inner product.

In [7]:
print("LangChain distance strategy:", vector_store.distance_strategy)
print("FAISS index type:", type(vector_store.index))
print("FAISS metric_type:", getattr(vector_store.index, "metric_type", None))

LangChain distance strategy: DistanceStrategy.MAX_INNER_PRODUCT
FAISS index type: <class 'faiss.swigfaiss_avx2.IndexFlatIP'>
FAISS metric_type: 0


## 7. Retrieve Relevant Context

Retrieval embeds the user query, searches the FAISS index, and returns the closest chunks with source metadata.

This is the evidence bundle that the LLM is allowed to use.

In [8]:
from src.retrievers.rag_retriever import retrieve_context

query = "Que canal conviene priorizar para una campana de awareness?"
retrieved_chunks = retrieve_context(vector_store, query, top_k=retrieval_settings.top_k)

for i, chunk in enumerate(retrieved_chunks, 1):
    print(f"\n{i}. similarity={chunk.similarity:.4f} raw_inner_product={chunk.score:.4f} source={chunk.source_label}")
    print(" ".join(chunk.text.split())[:700])


1. similarity=0.6165 raw_inner_product=0.6165 source=M5_Playbook_Brand_Awareness.pdf, page 0
18,192 +6 Impresiones 56,169 56,034 +135 Engagement Score 4.37 4.37 — Estrategia Multi-Canal para Brand Awareness Las campañas de Brand Awareness se benefician de la mayor cobertura posible. La distribución recomendada para este objetivo es: Instagram (35%) para contenido visual premium, Facebook (30%) para alcance en audiencias 35-60, Twitter (20%) para conversación en tiempo real, y Pinterest (15%) como canal de inspiración de largo plazo. Duración Óptima para Brand Awareness Las campañas de 45 días muestran la mejor relación costo-cobertura para Brand Awareness. Menos de 30 días no permite construir frecuencia suficiente (mínimo 7 impactos por usuario para memorización de marca). Más de 6

2. similarity=0.5566 raw_inner_product=0.5566 source=L1_Guia_Estrategica_Canales.pdf, page 2
2. Facebook: Canal de Alcance Masivo y Segmentación Precisa Facebook mantiene el mayor volumen de impresiones p

## 8. Ground The Answer Prompt

The answer chain receives only:

- the user question
- retrieved passages
- source metadata
- grounding instructions

The model should not answer from memory. If context is weak, it should say that the indexed documents do not contain enough information.

In [22]:
passages = [
    {"source": chunk.source_label, "text": chunk.text}
    for chunk in retrieved_chunks
]

In [23]:
passages

[{'source': 'M5_Playbook_Brand_Awareness.pdf, page 0',
  'text': '18,192\n+6\nImpresiones\n56,169\n56,034\n+135\nEngagement Score\n4.37\n4.37\n—\nEstrategia Multi-Canal para Brand Awareness\nLas campañas de Brand Awareness se benefician de la mayor cobertura posible. La distribución\nrecomendada para este objetivo es: Instagram (35%) para contenido visual premium, Facebook (30%)\npara alcance en audiencias 35-60, Twitter (20%) para conversación en tiempo real, y Pinterest (15%)\ncomo canal de inspiración de largo plazo.\nDuración Óptima para Brand Awareness\nLas campañas de 45 días muestran la mejor relación costo-cobertura para Brand Awareness. Menos\nde 30 días no permite construir frecuencia suficiente (mínimo 7 impactos por usuario para\nmemorización de marca). Más de 60 días incrementa el riesgo de fatiga sin ganancia proporcional en\nmétricas de reconocimiento.'},
 {'source': 'L1_Guia_Estrategica_Canales.pdf, page 2',
  'text': '2. Facebook: Canal de Alcance Masivo y Segmentación

In [9]:
from src.prompts.templates import GROUNDING_SYSTEM_PROMPT, GUIDELINE_STYLE_PROMPT, format_retrieved_context

passages = [
    {"source": chunk.source_label, "text": chunk.text}
    for chunk in retrieved_chunks
]

grounded_context = format_retrieved_context(passages)
print(GROUNDING_SYSTEM_PROMPT)
print("\n--- Style guidance ---\n")
print(GUIDELINE_STYLE_PROMPT)
print("\n--- Retrieved context preview ---\n")
print(grounded_context[:2000])

Eres un asistente documentado para inteligencia de campañas de marketing.
Responde en español y usa solo el contexto recuperado.
No inventes datos, citas, fuentes ni recomendaciones que no esten respaldadas por el contexto.
Si el contexto no alcanza para responder, dilo de forma breve y pide mas informacion o documentos.
Cuando sea posible, menciona los documentos fuente utilizados.

--- Style guidance ---

Estilo de respuesta:
- Se claro, ejecutivo y orientado a decisiones.
- Separa hechos recuperados de recomendaciones.
- Si das una recomendación, explica el criterio usado.
- No cites documentos de guidelines como fuentes de conocimiento del usuario.

--- Retrieved context preview ---

[Fuente 1: M5_Playbook_Brand_Awareness.pdf, page 0]
18,192
+6
Impresiones
56,169
56,034
+135
Engagement Score
4.37
4.37
—
Estrategia Multi-Canal para Brand Awareness
Las campañas de Brand Awareness se benefician de la mayor cobertura posible. La distribución
recomendada para este objetivo es: Instagram

## 9. Answering Provider Options

Retrieval is always local in this notebook: BGE-M3 embeddings plus FAISS cosine similarity.

Answer generation is optional and can use one of four routes:

- Qwen API: add QWEN_API_KEY or DASHSCOPE_API_KEY to .env, then select Qwen at runtime.
- OpenAI API: add OPENAI_API_KEY to .env, then select OpenAI at runtime.
- Local Hugging Face Transformers: select huggingface_local and a model at runtime.
- Ollama: run Ollama locally, pull a model, then select ollama and the model at runtime.

Do not change .env every time you want a different model. .env is only for secrets and endpoints.

If you have a Hugging Face token, add HF_TOKEN=... to .env. It is optional for public models, but useful for smoother downloads and higher rate limits.


In [10]:
from src.config.catalogs import available_answering_models_for, answering_models_for
from src.config.settings import configured_answering_providers

provider_names = ["qwen", "openai", "huggingface_local", "ollama"]
for provider_name in provider_names:
    print(f"\n{provider_name}")
    for model_spec in available_answering_models_for(provider_name):
        print(f"  - {model_spec.model} :: {model_spec.label}")

print("\nAvailable in this runtime:", configured_answering_providers())



qwen
  - qwen-plus :: Qwen Plus
  - qwen-turbo :: Qwen Turbo

openai
  - gpt-4o-mini :: OpenAI GPT-4o mini

huggingface_local
  - Qwen/Qwen2.5-0.5B-Instruct :: HF local Qwen2.5 0.5B Instruct (CPU-friendly)
  - Qwen/Qwen2.5-1.5B-Instruct :: HF local Qwen2.5 1.5B Instruct

ollama
  - llama3.2:3b :: Ollama llama3.2:3b
  - qwen2.5:3b :: Ollama qwen2.5:3b
  - qwen3:4b :: Ollama qwen3:4b (Arena-informed candidate)
  - gemma3:4b :: Ollama gemma3:4b (Arena-informed candidate)
  - granite3.2-vision:latest :: Ollama installed granite3.2-vision:latest
  - llama3:latest :: Ollama installed llama3:latest
  - llava:7b :: Ollama installed llava:7b

Available in this runtime: ['huggingface_local', 'ollama']


### Manual Provider Selection Example

Use this cell to choose a provider/model for the generation example. By default it stays retrieval-only, which makes the notebook safe to run even when API keys, Ollama, or local model weights are not ready.

Important: local Hugging Face models may download weights on first use. Ollama models must be pulled first, for example ollama pull qwen3:4b.


In [ ]:
# Leave these as None for a retrieval-only walkthrough.
# Change them when you want to test a specific answering route.
# Examples:
# provider, model = "qwen", "qwen-plus"
# provider, model = "openai", "gpt-4o-mini"
# provider, model = "huggingface_local", "Qwen/Qwen2.5-1.5B-Instruct"
# provider, model = "ollama", "qwen2.5:3b"

provider = None
model = None

provider, model


## 10. Optional: Generate A Grounded Answer

This step uses the provider and model selected above. If either is None, generation is skipped and the notebook remains retrieval-only.


In [15]:
from src.chains.grounded_qa import answer_with_context
from src.config.settings import AnsweringSettings, configured_answering_providers
from src.llms.factory import create_chat_provider

enabled_providers = configured_answering_providers()
print("Enabled providers:", enabled_providers)
print("Selected provider/model:", provider, model)


Enabled providers: ['huggingface_local', 'ollama']
Selected provider/model: ollama qwen2.5:3b


In [17]:
query

'Que canal conviene priorizar para una campana de awareness?'

In [16]:
if provider and model:
    if provider not in configured_answering_providers():
        print(f"Provider {provider!r} is not available in this runtime. Skipping generation.")
    else:
        answering = AnsweringSettings(provider=provider, model=model)
        llm = create_chat_provider(
            answering.provider,
            model=answering.model,
            temperature=answering.temperature,
        )
        result = answer_with_context(query=query, chunks=retrieved_chunks, llm=llm)
        print(result.answer)
        print("\nSources:")
        for source in result.sources:
            print("-", source["source"])
else:
    print("No provider/model selected. Skipping generation.")


Para una campaña de awareness, la recomendación es priorizar Instagram y Facebook en base a los datos proporcionados.

Instagram se destaca como canal de contenido visual premium para campañas de awareness con un engagement score de 4.37. Además, tiene una tasa de impresión promedio más alta que Twitter (61,430 vs 61,284).

Facebook es el canal de alcance masivo y segmentación precisa recomendado para campañas de awareness. Tiene un ROI promedio de 3.97x para estas campañas.

La estrategia multi-canal sugerida por la guía M5_Playbook_Brand_Awareness recomienda que Instagram (35%) y Facebook (30%) sean los canales principales, con Pinterest como canal secundario para inspiración de largo plazo.

Sources:
- M5_Playbook_Brand_Awareness.pdf, page 0
- L1_Guia_Estrategica_Canales.pdf, page 2
- L1_Guia_Estrategica_Canales.pdf, page 5
- M3_Guia_Campaign_Product_Launch.pdf, page 0


## 11. Where The CSV Fits

`Social_Media_Advertising.csv` is not part of the current FAISS PDF index.

Recommended next step:

1. Inspect the CSV schema.
2. Compute deterministic summaries with pandas.
3. Generate small summary documents or expose a structured analytics tool.
4. Combine retrieved PDF strategy context with CSV-derived metrics in the final answer.

This avoids pretending that raw rows are the same kind of knowledge as strategy documents.

In [ ]:
import pandas as pd

csv_path = PROJECT_ROOT / "data" / "corpus" / "Social_Media_Advertising.csv"
df_head = pd.read_csv(csv_path, nrows=5)
df_head